<a href="https://colab.research.google.com/github/Yash-k10/pattern_recognition/blob/main/practical7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================================
# FAST VEHICLE NUMBER PLATE VALIDATION PRACTICAL
# ============================================================
# Uses maximum 20 images from uploaded Kaggle ZIP
#
# Demonstrates:
# 1. OCR
# 2. String pattern recognition
# 3. Regular expressions
# 4. Production rules
# 5. Grammar validation
# 6. Parse tree
# ============================================================

# Install libraries
!pip install -q easyocr opencv-python-headless pandas matplotlib


# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import re
import cv2
import zipfile
import shutil
import random
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

import easyocr


# ============================================================
# 1. UPLOAD ZIP
# ============================================================

print("Upload your Kaggle dataset ZIP")

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

print("Uploaded:", zip_file)


# ============================================================
# 2. EXTRACT ZIP
# ============================================================

extract_path = "/content/dataset"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path)

print("\nExtracting dataset...")

with zipfile.ZipFile(
    zip_file,
    "r"
) as zip_ref:

    zip_ref.extractall(
        extract_path
    )

print("Extraction completed!")


# ============================================================
# 3. FIND IMAGES
# ============================================================

extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

all_images = []

for root, dirs, files_in_folder in os.walk(
    extract_path
):

    for file in files_in_folder:

        if file.lower().endswith(
            extensions
        ):

            all_images.append(
                os.path.join(
                    root,
                    file
                )
            )


print(
    "\nTotal images found:",
    len(all_images)
)


# ============================================================
# 4. SELECT ONLY 20 IMAGES
# ============================================================

random.seed(42)

if len(all_images) > 20:

    selected_images = random.sample(
        all_images,
        20
    )

else:

    selected_images = all_images


print(
    "Images selected for practical:",
    len(selected_images)
)


# ============================================================
# 5. LOAD OCR
# ============================================================

print("\nLoading OCR model...")

reader = easyocr.Reader(
    ["en"],
    gpu=True
)

print("OCR ready!")


# ============================================================
# 6. FAST OCR FUNCTION
# ============================================================

def extract_text(image_path):

    image = cv2.imread(
        image_path
    )

    if image is None:
        return ""


    # Resize image
    # Smaller image = faster OCR

    height, width = image.shape[:2]

    max_width = 800

    if width > max_width:

        scale = max_width / width

        image = cv2.resize(
            image,
            None,
            fx=scale,
            fy=scale
        )


    # OCR directly on image

    try:

        results = reader.readtext(
            image,
            detail=0,
            paragraph=False
        )

        text = "".join(
            results
        ).upper()

        # Keep only A-Z and 0-9

        text = re.sub(
            r"[^A-Z0-9]",
            "",
            text
        )

        return text

    except:

        return ""


# ============================================================
# 7. STRING VALIDATION
# ============================================================

def string_validation(
    plate
):

    plate = plate.upper()

    if len(plate) != 10:

        return False

    # XX00XX0000

    if not plate[0:2].isalpha():
        return False

    if not plate[2:4].isdigit():
        return False

    if not plate[4:6].isalpha():
        return False

    if not plate[6:10].isdigit():
        return False

    return True


# ============================================================
# 8. REGEX VALIDATION
# ============================================================

def regex_validation(
    plate
):

    pattern = (
        r"^[A-Z]{2}"
        r"[0-9]{2}"
        r"[A-Z]{2}"
        r"[0-9]{4}$"
    )

    return bool(
        re.fullmatch(
            pattern,
            plate
        )
    )


# ============================================================
# 9. GRAMMAR / PRODUCTION RULE VALIDATION
# ============================================================

def grammar_validation(
    plate
):

    # Production rule:
    #
    # NUMBER_PLATE
    #       ↓
    # STATE_CODE RTO_CODE SERIES DIGITS
    #
    # STATE_CODE → LETTER LETTER
    # RTO_CODE   → DIGIT DIGIT
    # SERIES     → LETTER LETTER
    # DIGITS     → DIGIT DIGIT DIGIT DIGIT


    if len(plate) != 10:
        return False


    # STATE_CODE

    if not (
        plate[0].isalpha()
        and
        plate[1].isalpha()
    ):
        return False


    # RTO_CODE

    if not (
        plate[2].isdigit()
        and
        plate[3].isdigit()
    ):
        return False


    # SERIES

    if not (
        plate[4].isalpha()
        and
        plate[5].isalpha()
    ):
        return False


    # DIGITS

    if not all(
        c.isdigit()
        for c in plate[6:10]
    ):
        return False


    return True


# ============================================================
# 10. PARSE TREE
# ============================================================

class Node:

    def __init__(
        self,
        value
    ):

        self.value = value
        self.children = []


def create_tree(
    plate
):

    if not grammar_validation(
        plate
    ):

        return None


    root = Node(
        "NUMBER_PLATE"
    )


    # STATE CODE

    state = Node(
        "STATE_CODE"
    )

    state.children = [
        Node(
            f"LETTER({plate[0]})"
        ),
        Node(
            f"LETTER({plate[1]})"
        )
    ]


    # RTO CODE

    rto = Node(
        "RTO_CODE"
    )

    rto.children = [
        Node(
            f"DIGIT({plate[2]})"
        ),
        Node(
            f"DIGIT({plate[3]})"
        )
    ]


    # SERIES

    series = Node(
        "SERIES"
    )

    series.children = [
        Node(
            f"LETTER({plate[4]})"
        ),
        Node(
            f"LETTER({plate[5]})"
        )
    ]


    # DIGITS

    digits = Node(
        "DIGITS"
    )

    for d in plate[6:10]:

        digits.children.append(
            Node(
                f"DIGIT({d})"
            )
        )


    root.children = [
        state,
        rto,
        series,
        digits
    ]


    return root


# ============================================================
# 11. PRINT TREE
# ============================================================

def print_tree(
    node,
    level=0
):

    print(
        "    " * level
        + "└── "
        + node.value
    )

    for child in node.children:

        print_tree(
            child,
            level + 1
        )


# ============================================================
# 12. PROCESS ONLY 20 IMAGES
# ============================================================

print("\nProcessing images...")

results = []

for image_path in selected_images:

    # OCR

    text = extract_text(
        image_path
    )


    # Validation

    string_result = \
        string_validation(
            text
        )

    regex_result = \
        regex_validation(
            text
        )

    grammar_result = \
        grammar_validation(
            text
        )


    results.append({

        "Image":
            os.path.basename(
                image_path
            ),

        "OCR_Text":
            text,

        "String":
            string_result,

        "Regex":
            regex_result,

        "Grammar":
            grammar_result,

        "Final_Result":
            "VALID"
            if grammar_result
            else "INVALID"
    })


# ============================================================
# 13. RESULTS
# ============================================================

df = pd.DataFrame(
    results
)


print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

display(df)


# ============================================================
# 14. SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    "Images processed:",
    len(df)
)

print(
    "String valid:",
    df["String"].sum()
)

print(
    "Regex valid:",
    df["Regex"].sum()
)

print(
    "Grammar valid:",
    df["Grammar"].sum()
)


# ============================================================
# 15. PARSE TREE
# ============================================================

valid_rows = df[
    df["Grammar"] == True
]


print("\n" + "=" * 70)
print("PARSE TREE")
print("=" * 70)


if len(valid_rows) > 0:

    plate = valid_rows.iloc[0][
        "OCR_Text"
    ]

    print(
        "Detected plate:",
        plate
    )

    print()

    tree = create_tree(
        plate
    )

    print_tree(
        tree
    )

else:

    print(
        "No valid plate detected."
    )


# ============================================================
# 16. TEST WITH KNOWN VALID/INVALID PLATES
# ============================================================

print("\n" + "=" * 70)
print("PRODUCTION RULE DEMONSTRATION")
print("=" * 70)


examples = [
    "MH12AB1234",
    "DL01CD5678",
    "KA05EF4321",
    "MH12AB123",
    "MH1AAB1234",
    "MH12AB12CD"
]


for plate in examples:

    print(
        plate,
        "→",
        "VALID"
        if grammar_validation(
            plate
        )
        else "INVALID"
    )


# ============================================================
# 17. SAVE CSV
# ============================================================

output_file = (
    "/content/"
    "number_plate_results.csv"
)

df.to_csv(
    output_file,
    index=False
)

print(
    "\nResults saved to:",
    output_file
)


# ============================================================
# 18. DOWNLOAD CSV
# ============================================================

files.download(
    output_file
)

print(
    "\nPRACTICAL COMPLETED!"
)

Upload your Kaggle dataset ZIP


Saving number_plate_dataset_25.zip to number_plate_dataset_25.zip
Uploaded: number_plate_dataset_25.zip

Extracting dataset...
Extraction completed!

Total images found: 25
Images selected for practical: 20

Loading OCR model...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteOCR ready!

Processing images...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



VALIDATION RESULTS


,Image,OCR_Text,String,Regex,Grammar,Final_Result
0,plate_06.png,INDMPO9KL1357REGISTRATION,False,False,False,INVALID
1,plate_05.png,INDGJ01IJ4321REGISTRATION,False,False,False,INVALID
2,plate_21.png,INDUKOZPQ9281REGISTRATION,False,False,False,INVALID
3,plate_25.png,INDMHB1XY8062REGISTRATION,False,False,False,INVALID
4,plate_23.png,INDHP12TU7836REGISTRATION,False,False,False,INVALID
5,plate_02.png,INDMH14CD5678REGISTRATION,False,False,False,INVALID
6,plate_03.png,INDDLO1EF2345REGISTRATION,False,False,False,INVALID
7,plate_07.png,INDRJ14MN2468REGISTRATION,False,False,False,INVALID
8,plate_01.png,INDMH1ZAB1234REGISTRATION,False,False,False,INVALID
9,plate_10.png,INDWBO6ST5319REGISTRATION,False,False,False,INVALID



SUMMARY
Images processed: 20
String valid: 0
Regex valid: 0
Grammar valid: 0

PARSE TREE
No valid plate detected.

PRODUCTION RULE DEMONSTRATION
MH12AB1234 → VALID
DL01CD5678 → VALID
KA05EF4321 → VALID
MH12AB123 → INVALID
MH1AAB1234 → INVALID
MH12AB12CD → INVALID

Results saved to: /content/number_plate_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PRACTICAL COMPLETED!
